<a href="https://colab.research.google.com/github/2anoFIAP/AED-KPOP-CP-DATASCIENCE/blob/main/AEDTeste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dados = []

regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
frete_base = {
    "Sudeste": 12,
    "Sul": 17,
    "Nordeste": 29,
    "Centro-Oeste": 22,
    "Norte": 38
}

pedido_id = 1

for cliente_id in range(1, 4001):
    regiao = str(rng.choice(regioes))
    perfil_cupom = float(rng.choice([0.1, 0.5, 0.9]))

    primeira_compra = (
        pd.Timestamp("2025-01-01")
        + pd.Timedelta(days=int(rng.integers(0, 220)))
    )

    # Pedidos iniciais e possíveis recompras futuras
    dias = [0]

    if rng.random() < 0.65:
        dias.append(int(rng.integers(1, 30)))

    media_recompras = 1.5 if perfil_cupom < 0.9 else 0.9
    qtd_futura = int(rng.poisson(media_recompras))
    dias.extend(rng.integers(30, 120, size=qtd_futura))

    for dia in dias:
        modalidade = str(rng.choice(
            ["ultra", "rapida", "normal"],
            p=[0.30, 0.40, 0.30]
        ))

        prazo = {
            "ultra": int(rng.integers(1, 3)),
            "rapida": int(rng.integers(3, 6)),
            "normal": int(rng.integers(6, 10))
        }[modalidade]

        receita = round(float(rng.gamma(4, 60) + 30), 2)
        cupom = bool(rng.random() < perfil_cupom)

        desconto = round(
            receita * float(rng.uniform(0.08, 0.25))
            if cupom else 0,
            2
        )

        custo_produto = round(
            receita * float(rng.uniform(0.47, 0.68)), 2
        )

        adicional = {
            "ultra": 20,
            "rapida": 9,
            "normal": 0
        }[modalidade]

        custo_logistico = round(
            frete_base[regiao]
            + adicional
            + float(rng.uniform(0, 8)),
            2
        )

        nota_base = {
            "ultra": 4.6,
            "rapida": 4.5,
            "normal": 3.95
        }[modalidade]

        avaliacao = round(
            float(np.clip(rng.normal(nota_base, 0.5), 1, 5)),
            1
        )

        dados.append({
            "pedido_id": pedido_id,
            "cliente_id": cliente_id,
            "data_pedido": primeira_compra + pd.Timedelta(days=int(dia)),
            "regiao": regiao,
            "receita_bruta": receita,
            "desconto_valor": desconto,
            "custo_produto": custo_produto,
            "custo_logistico": custo_logistico,
            "prazo_entrega_dias": prazo,
            "avaliacao_cliente": avaliacao,
            "cupom_aplicado": cupom
        })

        pedido_id += 1

df = pd.DataFrame(dados)

# Simular avaliações ausentes
indices = rng.choice(
    df.index,
    size=int(len(df) * 0.03),
    replace=False
)
df.loc[indices, "avaliacao_cliente"] = np.nan

print("Pedidos gerados:", len(df))
print("Clientes:", df["cliente_id"].nunique())
print("Regiões:", df["regiao"].nunique())

display(df.head())

# Salvar dataset para utilizar posteriormente
df.to_csv("ecommerce_sintetico.csv", index=False)


Pedidos gerados: 11767
Clientes: 4000
Regiões: 5


,pedido_id,cliente_id,data_pedido,regiao,receita_bruta,desconto_valor,custo_produto,custo_logistico,prazo_entrega_dias,avaliacao_cliente,cupom_aplicado
0,1,1,2025-05-25,Sudeste,248.08,58.93,150.14,18.58,8,4.2,True
1,2,1,2025-08-02,Sudeste,157.20,32.84,85.59,39.77,2,4.5,True
2,3,2,2025-06-17,Centro-Oeste,206.53,33.01,105.29,43.04,2,4.2,True
3,4,2,2025-06-23,Centro-Oeste,346.92,46.18,223.69,48.44,2,4.7,True
4,5,3,2025-02-22,Norte,121.46,0.00,71.59,59.12,2,5.0,False
